# Kaggle Project: Residential Property Value Prediction

In [ ]:
# Connect to google colab
# skip it if not using colab
try:
  from google.colab import drive, userdata
  drive.mount('/content/drive', force_remount=True)
  COLAB = True
  print("Note: using Google CoLab")
except:
  print("Note: not using Google CoLab")
  COLAB = False

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 6.3 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-jxk0warx
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-jxk0warx
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.2 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=f135def12da2a031f9645d718dfcd760656d759e9b43b539ea5c23e42542e539
  Stored in directory: /tmp/pip-ephem-wheel-cache-d756l6xr/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip
Mounted at /content/drive
Note: using Google CoLab
Using device: cpu


## Import Packages and Load Data

In [ ]:
# ============================================================
# Import Packages
# ============================================================
import os
import numpy as np
import pandas as pd
from typing import List, Optional
from sklearn.model_selection import train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
# install packages if not already installed
# ! pip install catboost
# ! pip install git+https://github.com/openai/CLIP.git
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import clip
from PIL import Image
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer

# ============================================================
# Load Data
# ============================================================

# change it to your own path
train_path = "/content/drive/My Drive/Colab Notebooks/train.csv"
test_path  = "/content/drive/My Drive/Colab Notebooks/test.csv"
IMG_DIR = "/content/drive/My Drive/Colab Notebooks/imgs"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## Data Cleanning

In [ ]:
# ============================================================
# 1. Data Cleanning
# ============================================================

df      = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

id_col     = df.columns[0]
target_col = df.columns[-1]

print("ID column    :", id_col)
print("Target column:", target_col)

# get rid of na value
y_raw = df[target_col].replace([np.inf, -np.inf], np.nan)
keep  = y_raw.notna()
dropped = (~keep).sum()
if dropped:
    print(f"⚠ Dropping {dropped} rows with invalid target")
df = df.loc[keep].reset_index(drop=True)

X = df.drop(columns=[id_col, target_col])
y = df[target_col].astype(float)
X_test_raw = df_test.drop(columns=[id_col])

# set random seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# train/val split
idx_all = np.arange(len(df))
idx_train, idx_val = train_test_split(
    idx_all,
    test_size=0.2,
    random_state=RANDOM_STATE
)

X_train_tab = X.iloc[idx_train].copy()
X_val_tab   = X.iloc[idx_val].copy()
y_train     = y.iloc[idx_train]
y_val       = y.iloc[idx_val]


ID column    : id
Target column: property_value_score


## Tree Methods

### Data Preprocessing

In [ ]:
# ============================================================
# Data Preporcessing for Tree Models
# ============================================================

y_all = y.values
N = len(df)
N_test = len(df_test)
print("N train:", N, "N test:", N_test)

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
print("Numeric cols :", len(num_cols))
print("Categorical  :", len(cat_cols))

preprocess_tree = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline(steps=[
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ]
)

X_all_tree_full = preprocess_tree.fit_transform(X)
X_test_tree     = preprocess_tree.transform(X_test_raw)

if hasattr(X_all_tree_full, "toarray"):
    X_all_tree_full = X_all_tree_full.toarray().astype(np.float32)
    X_test_tree     = X_test_tree.toarray().astype(np.float32)
else:
    X_all_tree_full = X_all_tree_full.astype(np.float32)
    X_test_tree     = X_test_tree.astype(np.float32)

X_train_tree = X_all_tree_full[idx_train]
X_val_tree   = X_all_tree_full[idx_val]

print("X_all_tree_full:", X_all_tree_full.shape)
print("X_train_tree   :", X_train_tree.shape)
print("X_val_tree     :", X_val_tree.shape)
print("X_test_tree    :", X_test_tree.shape)

N train: 8000 N test: 2000
Numeric cols : 8
Categorical  : 23
X_all_tree_full: (8000, 10396)
X_train_tree   : (6400, 10396)
X_val_tree     : (1600, 10396)
X_test_tree    : (2000, 10396)


### XGBoost

In [ ]:
# ============================================================
# XGBoost
# ============================================================
oof_xgb = np.zeros(N, dtype=np.float32)
test_xgb_folds = []

params_xgb = {
    "objective": "reg:squarederror",
    "eta": 0.02,
    "max_depth": 8,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "lambda": 1.0,
    "alpha": 1.0,
    "eval_metric": "rmse",
}

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all_tree_full), 1):
    print(f"\n[OOF XGB] Fold {fold}/5")
    X_tr, X_va = X_all_tree_full[tr_idx], X_all_tree_full[va_idx]
    y_tr, y_va = y_all[tr_idx], y_all[va_idx]

    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval   = xgb.DMatrix(X_va, label=y_va)
    dtest  = xgb.DMatrix(X_test_tree)

    evals = [(dtrain, "train"), (dval, "val")]

    model_xgb = xgb.train(
        params_xgb,
        dtrain,
        num_boost_round=2000,
        evals=evals,
        early_stopping_rounds=50,
        verbose_eval=False,
    )

    pred_va = model_xgb.predict(dval)
    oof_xgb[va_idx] = pred_va

    pred_test_fold = model_xgb.predict(dtest)
    test_xgb_folds.append(pred_test_fold)

    rmse_fold = np.sqrt(mean_squared_error(y_va, pred_va))
    print(f"[OOF XGB] Fold {fold} RMSE: {rmse_fold:.4f}")

test_xgb_folds = np.stack(test_xgb_folds, axis=0) 
pred_xgb_test_oof = test_xgb_folds.mean(axis=0)

rmse_xgb_oof = np.sqrt(mean_squared_error(y_all, oof_xgb))
print("\n[OOF XGB] Overall OOF RMSE on full train:", rmse_xgb_oof)


[OOF XGB] Fold 1/5
[OOF XGB] Fold 1 RMSE: 3.5173

[OOF XGB] Fold 2/5
[OOF XGB] Fold 2 RMSE: 3.3093

[OOF XGB] Fold 3/5
[OOF XGB] Fold 3 RMSE: 3.5854

[OOF XGB] Fold 4/5
[OOF XGB] Fold 4 RMSE: 3.5017

[OOF XGB] Fold 5/5
[OOF XGB] Fold 5 RMSE: 3.4691

[OOF XGB] Overall OOF RMSE on full train: 3.4777744087388704


### LightGBM

In [ ]:
# ============================================================
# LightGBM
# ============================================================

oof_lgb = np.zeros(N, dtype=np.float32)
test_lgb_folds = []

params_lgb = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.02,
    "num_leaves": 128,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 1.0,
    "lambda_l2": 1.0,
    "max_depth": -1,
    "verbosity": -1,
}

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all_tree_full), 1):
    print(f"\n[OOF LGB] Fold {fold}/5")
    X_tr, X_va = X_all_tree_full[tr_idx], X_all_tree_full[va_idx]
    y_tr, y_va = y_all[tr_idx], y_all[va_idx]

    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data   = lgb.Dataset(X_va, label=y_va)

    early_stop_cb = lgb.early_stopping(stopping_rounds=100, verbose=False)

    model_lgb = lgb.train(
        params_lgb,
        train_data,
        num_boost_round=5000,
        valid_sets=[val_data],
        callbacks=[early_stop_cb],
    )

    pred_va = model_lgb.predict(X_va, num_iteration=model_lgb.best_iteration)
    oof_lgb[va_idx] = pred_va

    pred_test_fold = model_lgb.predict(X_test_tree, num_iteration=model_lgb.best_iteration)
    test_lgb_folds.append(pred_test_fold)

    rmse_fold = np.sqrt(mean_squared_error(y_va, pred_va))
    print(f"[OOF LGB] Fold {fold} RMSE: {rmse_fold:.4f}")

test_lgb_folds = np.stack(test_lgb_folds, axis=0)
pred_lgb_test_oof = test_lgb_folds.mean(axis=0)

rmse_lgb_oof = np.sqrt(mean_squared_error(y_all, oof_lgb))
print("\n[OOF LGB] Overall OOF RMSE on full train:", rmse_lgb_oof)


[OOF LGB] Fold 1/5
[OOF LGB] Fold 1 RMSE: 3.4823

[OOF LGB] Fold 2/5
[OOF LGB] Fold 2 RMSE: 3.3313

[OOF LGB] Fold 3/5
[OOF LGB] Fold 3 RMSE: 3.5563

[OOF LGB] Fold 4/5
[OOF LGB] Fold 4 RMSE: 3.5032

[OOF LGB] Fold 5/5
[OOF LGB] Fold 5 RMSE: 3.4512

[OOF LGB] Overall OOF RMSE on full train: 3.4656849662911577


### CatBoost

In [ ]:
# ============================================================
# CatBoost
# ============================================================

oof_cat = np.zeros(N, dtype=np.float32)
test_cat_folds = []

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all_tree_full), 1):
    print(f"\n[OOF CatBoost] Fold {fold}/5")
    X_tr, X_va = X_all_tree_full[tr_idx], X_all_tree_full[va_idx]
    y_tr, y_va = y_all[tr_idx], y_all[va_idx]

    model_cat = CatBoostRegressor(
        iterations=1500,
        depth=8,
        learning_rate=0.03,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=RANDOM_STATE,
        task_type="CPU",    # switch to GPU if using CUDA
        verbose=False
    )

    model_cat.fit(X_tr, y_tr, eval_set=(X_va, y_va))

    pred_va = model_cat.predict(X_va)
    oof_cat[va_idx] = pred_va

    pred_test_fold = model_cat.predict(X_test_tree)
    test_cat_folds.append(pred_test_fold)

    rmse_fold = np.sqrt(mean_squared_error(y_va, pred_va))
    print(f"[OOF CatBoost] Fold {fold} RMSE: {rmse_fold:.4f}")

test_cat_folds = np.stack(test_cat_folds, axis=0)
pred_cat_test_oof = test_cat_folds.mean(axis=0)

rmse_cat_oof = np.sqrt(mean_squared_error(y_all, oof_cat))
print("\n[OOF CatBoost] Overall OOF RMSE on full train:", rmse_cat_oof)


[OOF CatBoost] Fold 1/5
[OOF CatBoost] Fold 1 RMSE: 3.3823

[OOF CatBoost] Fold 2/5
[OOF CatBoost] Fold 2 RMSE: 3.2519

[OOF CatBoost] Fold 3/5
[OOF CatBoost] Fold 3 RMSE: 3.4315

[OOF CatBoost] Fold 4/5
[OOF CatBoost] Fold 4 RMSE: 3.3773

[OOF CatBoost] Fold 5/5
[OOF CatBoost] Fold 5 RMSE: 3.3468

[OOF CatBoost] Overall OOF RMSE on full train: 3.358473136180654


## MultiModal

### Load Image Data

In [ ]:
# ============================================================
# Load Image Data
# ============================================================

img_dir = Path(IMG_DIR)
train_ids_full = df[id_col].values
test_ids       = df_test[id_col].values

# CLIP extract features
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()

class CLIPImageDataset(Dataset):
    def __init__(self, ids: np.ndarray):
        self.ids = ids
        self.paths: List[Path] = []
        for pid in self.ids:
            p = img_dir / f"{int(pid)}.jpg"
            self.paths.append(p)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        pid = self.ids[idx]
        img_path = self.paths[idx]
        if img_path.exists():
            try:
                img = Image.open(img_path).convert("RGB")
                img = clip_preprocess(img)
            except Exception:
                img = torch.zeros(3, 224, 224)
        else:
            img = torch.zeros(3, 224, 224)
        return pid, img

def extract_clip_feats(ids: np.ndarray, batch_size: int = 64):
    ds = CLIPImageDataset(ids)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)
    feat_dict = {}
    with torch.no_grad():
        for pids, imgs in dl:
            imgs = imgs.to(device)
            feats = clip_model.encode_image(imgs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            feats = feats.cpu().numpy().astype(np.float32)
            for pid, f in zip(pids.numpy(), feats):
                feat_dict[int(pid)] = f
    out = np.stack(
        [feat_dict.get(int(pid), np.zeros(512, dtype=np.float32)) for pid in ids],
        axis=0
    )
    return out

print("Extracting CLIP features for train...")
clip_train_feats_full = extract_clip_feats(train_ids_full)
print("Extracting CLIP features for test ...")
clip_test_feats = extract_clip_feats(test_ids)
print("clip_train_feats_full:", clip_train_feats_full.shape)
print("clip_test_feats:", clip_test_feats.shape)


========== MultiModal 5-fold KFold ==========


100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 142MiB/s]


Extracting CLIP features for train...
Extracting CLIP features for test ...
clip_train_feats_full: (8000, 512)
clip_test_feats      : (2000, 512)


### Define MultiModalMLP and other classes

In [ ]:
# ============================================================
# Define MultiModalMLP and other classes
# ============================================================

# Define MultiModelMLP
class MultiModalMLP(nn.Module):
    def __init__(self, tab_dim, img_dim=512, img_proj_dim=256):
        super().__init__()
        self.tab_branch = nn.Sequential(
            nn.Linear(tab_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
        )
        self.img_branch = nn.Sequential(
            nn.Linear(img_dim, img_proj_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.fusion = nn.Sequential(
            nn.Linear(128 + img_proj_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, tab_x, img_x):
        t = self.tab_branch(tab_x)
        i = self.img_branch(img_x)
        x = torch.cat([t, i], dim=1)
        return self.fusion(x)

# Define Dataset
class MMTabImgDataset(Dataset):
    def __init__(self, tab, img, targets=None):
        self.tab = tab.astype(np.float32)
        self.img = img.astype(np.float32)
        self.targets = targets.astype(np.float32) if targets is not None else None

    def __len__(self):
        return len(self.tab)

    def __getitem__(self, idx):
        out = {
            "tabular": torch.from_numpy(self.tab[idx]),
            "image":   torch.from_numpy(self.img[idx]),
        }
        if self.targets is not None:
            out["target"] = torch.tensor(self.targets[idx], dtype=torch.float32)
        return out

# EarlyStopping
class EarlyStopping:
    def __init__(self, patience=8, min_delta=1e-3, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_model = None
        self.best_loss = None
        self.counter = 0
        self.status = ""

    def __call__(self, model, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0
            self.status = "Initial best model set."
            return False
        elif self.best_loss - val_loss >= self.min_delta:
            self.best_loss = val_loss
            self.best_model = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0
            self.status = f"Improved to {val_loss:.4f}, counter reset."
            return False
        else:
            self.counter += 1
            self.status = f"No improvement, counter = {self.counter}"
            if self.counter >= self.patience:
                if self.restore_best_weights and self.best_model is not None:
                    model.load_state_dict(self.best_model)
                self.status = f"Early stopping at counter {self.counter}, best_loss={self.best_loss:.4f}"
                return True
            return False

### Train MultiModal

In [ ]:
# ============================================================
# Train MultiModal
# ============================================================

X_tab_all_mm = X_all_tree_full 
X_img_all_mm = clip_train_feats_full 
y_all = y.values 

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

mm_oof = np.zeros(len(df), dtype=np.float32)
mm_test_folds = []
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_tab_all_mm), start=1):
    print(f"\n===== MultiModal Fold {fold}/5 =====")

    X_tr_tab_raw = X_tab_all_mm[tr_idx]
    X_va_tab_raw = X_tab_all_mm[va_idx]
    X_tr_img = X_img_all_mm[tr_idx]
    X_va_img = X_img_all_mm[va_idx]
    y_tr     = y_all[tr_idx]
    y_va     = y_all[va_idx]

    scaler = RobustScaler()
    X_tr_tab = scaler.fit_transform(X_tr_tab_raw)
    X_va_tab = scaler.transform(X_va_tab_raw)
    X_test_tab = scaler.transform(X_test_tree) 

    train_ds = MMTabImgDataset(X_tr_tab, X_tr_img, y_tr)
    val_ds   = MMTabImgDataset(X_va_tab, X_va_img, y_va)
    test_ds  = MMTabImgDataset(X_test_tab, clip_test_feats, None)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=0)

    model_mm = MultiModalMLP(
        tab_dim=X_tr_tab.shape[1],
        img_dim=X_img_all_mm.shape[1]
    ).to(device)

    criterion_mm = nn.MSELoss()
    optimizer_mm = optim.Adam(model_mm.parameters(), lr=1e-3, weight_decay=1e-4)
    early_stop   = EarlyStopping(patience=8, min_delta=1e-3, restore_best_weights=True)

    max_epochs = 80
    for epoch in range(max_epochs):
        # train
        model_mm.train()
        train_loss = 0.0
        for batch in train_loader:
            tab = batch["tabular"].to(device)
            img = batch["image"].to(device)
            tgt = batch["target"].to(device).view(-1, 1)

            optimizer_mm.zero_grad()
            pred = model_mm(tab, img)
            loss = criterion_mm(pred, tgt)
            loss.backward()
            optimizer_mm.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # val
        model_mm.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                tab = batch["tabular"].to(device)
                img = batch["image"].to(device)
                tgt = batch["target"].to(device).view(-1, 1)
                pred = model_mm(tab, img)
                val_loss += criterion_mm(pred, tgt).item()
        val_loss /= len(val_loader)

        stop = early_stop(model_mm, val_loss)

        if (epoch + 1) % 10 == 0 or stop:
            print(f"[Fold {fold} Epoch {epoch+1}/{max_epochs}] "
                  f"train={train_loss:.4f}, val={val_loss:.4f} | {early_stop.status}")
        if stop:
            break

    # OOF prediction
    model_mm.eval()
    va_preds = []
    with torch.no_grad():
        for batch in val_loader:
            tab = batch["tabular"].to(device)
            img = batch["image"].to(device)
            pred = model_mm(tab, img).cpu().numpy().reshape(-1)
            va_preds.append(pred)
    va_preds = np.concatenate(va_preds)
    mm_oof[va_idx] = va_preds

    rmse_fold = np.sqrt(mean_squared_error(y_va, va_preds))
    fold_scores.append(rmse_fold)
    print(f"Fold {fold} RMSE: {rmse_fold:.4f}")

    # test prediction
    test_preds = []
    with torch.no_grad():
        for batch in test_loader:
            tab = batch["tabular"].to(device)
            img = batch["image"].to(device)
            pred = model_mm(tab, img).cpu().numpy().reshape(-1)
            test_preds.append(pred)
    test_preds = np.concatenate(test_preds)
    mm_test_folds.append(test_preds)

print("\nMultiModal 5-fold OOF RMSE (overall):",
      np.sqrt(mean_squared_error(y_all, mm_oof)))
print("Per-fold RMSE:", fold_scores)

mm_test_folds = np.stack(mm_test_folds, axis=0)
pred_mm_test  = mm_test_folds.mean(axis=0)

pred_mm_val = mm_oof[idx_val]
rmse_mm_val = np.sqrt(mean_squared_error(y_val.values, pred_mm_val))
print("MultiModal OOF RMSE on val idx:", rmse_mm_val)


===== MultiModal Fold 1/5 =====
[Fold 1 Epoch 10/80] train=19.5551, val=14.0394 | No improvement, counter = 4
[Fold 1 Epoch 14/80] train=17.8836, val=14.4983 | Early stopping at counter 8, best_loss=13.4744
Fold 1 RMSE: 3.6707

===== MultiModal Fold 2/5 =====
[Fold 2 Epoch 10/80] train=19.1517, val=14.4976 | No improvement, counter = 2
[Fold 2 Epoch 20/80] train=17.4705, val=13.9705 | No improvement, counter = 4
[Fold 2 Epoch 30/80] train=16.7703, val=16.7670 | No improvement, counter = 4
[Fold 2 Epoch 40/80] train=16.2382, val=13.1009 | No improvement, counter = 6
[Fold 2 Epoch 42/80] train=16.3267, val=14.0778 | Early stopping at counter 8, best_loss=12.8755
Fold 2 RMSE: 3.5882

===== MultiModal Fold 3/5 =====
[Fold 3 Epoch 10/80] train=19.4034, val=14.6355 | No improvement, counter = 6
[Fold 3 Epoch 12/80] train=18.5350, val=14.0386 | Early stopping at counter 8, best_loss=14.0286
Fold 3 RMSE: 3.7455

===== MultiModal Fold 4/5 =====
[Fold 4 Epoch 10/80] train=19.4894, val=14.6313 |

## TF-IDF for description

In [ ]:
# ============================================================
# TF-IDF
# ============================================================

DESC_COL = "description"

desc_all  = df[DESC_COL].fillna("").astype(str).values
desc_test = df_test[DESC_COL].fillna("").astype(str).values

oof_text = np.zeros(N, dtype=np.float32)
test_text_folds = []

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for fold, (tr_idx, va_idx) in enumerate(kf.split(desc_all), 1):
    print(f"\n[OOF TEXT] Fold {fold}/5")

    desc_tr = desc_all[tr_idx]
    desc_va = desc_all[va_idx]

    tfidf = TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 3),
        stop_words="english",
        min_df=3,
        max_df=0.9
    )

    X_tr_txt = tfidf.fit_transform(desc_tr)
    X_va_txt = tfidf.transform(desc_va)
    X_test_txt = tfidf.transform(desc_test)

    y_tr, y_va = y_all[tr_idx], y_all[va_idx]

    train_data_txt = lgb.Dataset(X_tr_txt, label=y_tr)
    val_data_txt   = lgb.Dataset(X_va_txt, label=y_va)

    params_txt = {
        "objective": "regression",
        "metric": "rmse",
        "learning_rate": 0.05,
        "num_leaves": 64,
        "feature_fraction": 0.7,
        "bagging_fraction": 0.7,
        "bagging_freq": 1,
        "lambda_l1": 0.0,
        "lambda_l2": 1.0,
        "max_depth": -1,
        "verbosity": -1,
    }

    early_stop_cb_txt = lgb.early_stopping(stopping_rounds=100, verbose=False)

    model_txt = lgb.train(
        params_txt,
        train_data_txt,
        num_boost_round=5000,
        valid_sets=[val_data_txt],
        callbacks=[early_stop_cb_txt]
    )

    pred_va = model_txt.predict(X_va_txt, num_iteration=model_txt.best_iteration)
    oof_text[va_idx] = pred_va

    pred_test_fold = model_txt.predict(X_test_txt, num_iteration=model_txt.best_iteration)
    test_text_folds.append(pred_test_fold)

    rmse_fold = np.sqrt(mean_squared_error(y_va, pred_va))
    print(f"[OOF TEXT] Fold {fold} RMSE: {rmse_fold:.4f}")

test_text_folds = np.stack(test_text_folds, axis=0)
pred_text_test_oof = test_text_folds.mean(axis=0)

rmse_text_oof = np.sqrt(mean_squared_error(y_all, oof_text))
print("\n[OOF TEXT] Overall OOF RMSE on full train:", rmse_text_oof)


[OOF TEXT] Fold 1/5
[OOF TEXT] Fold 1 RMSE: 4.2649

[OOF TEXT] Fold 2/5
[OOF TEXT] Fold 2 RMSE: 4.3876

[OOF TEXT] Fold 3/5
[OOF TEXT] Fold 3 RMSE: 4.4960

[OOF TEXT] Fold 4/5
[OOF TEXT] Fold 4 RMSE: 4.2606

[OOF TEXT] Fold 5/5
[OOF TEXT] Fold 5 RMSE: 4.2650

[OOF TEXT] Overall OOF RMSE on full train: 4.335838410452847


## Stacking

In [ ]:
# ============================================================
# Stacking with different models
# ============================================================

stack_oof = np.vstack([
    oof_xgb,
    oof_lgb,
    oof_cat,
    mm_oof,
    oof_text
]).T 

stack_test_oof = np.vstack([
    pred_xgb_test_oof,
    pred_lgb_test_oof,
    pred_cat_test_oof,
    pred_mm_test,
    pred_text_test_oof
]).T

print("stack_oof shape:", stack_oof.shape)
print("stack_test_oof shape:", stack_test_oof.shape)

stack_val = stack_oof[idx_val]
y_val_true = y_all[idx_val]

scaler_meta = StandardScaler()
stack_val_s  = scaler_meta.fit_transform(stack_val)
stack_test_s = scaler_meta.transform(stack_test_oof)

meta = Ridge(alpha=0.1, random_state=RANDOM_STATE)
meta.fit(stack_val_s, y_val_true)

val_pred_meta = meta.predict(stack_val_s)
rmse_meta_val = np.sqrt(mean_squared_error(y_val_true, val_pred_meta))
print(f"\n[Meta Ridge OOF] Val RMSE on idx_val: {rmse_meta_val:.4f}")
print("Meta weights (xgb, lgb, cat, mm, text):", meta.coef_)

len idx_val: 1600
stack_oof shape: (8000, 5)
stack_test_oof shape: (2000, 5)

[Meta Ridge OOF] Val RMSE on idx_val: 3.2588
Meta weights (xgb, lgb, cat, mm, text): [-2.554386   1.2300969  5.6445384  1.1079726  1.6763798]


## Submission

In [ ]:
# ============================================================
# Submission to Kaggle
# ============================================================

final_test_pred = meta.predict(stack_test_s)
final_test_pred = np.clip(final_test_pred, 0, 100)

submission_oof = pd.DataFrame({
    "id": df_test[id_col].values,
    "property_value_score": final_test_pred
})

submission_oof.to_csv("submission_oof_stacking1.csv", index=False)
print("✅ submission_oof_stacking.csv saved!", submission_oof.shape)
print(submission_oof.head())

✅ submission_oof_stacking.csv saved! (2000, 2)
     id  property_value_score
0  8001             61.055963
1  8002             54.371217
2  8003             57.765032
3  8004             64.263560
4  8005             62.627426
